# Experiments

Notebook harness for `predict.py`. Loads Qwen from `runtime.json`'s
revision, exposes it as an OpenAI-compatible endpoint on localhost, and
runs the experiments defined in `predict.py`.

## Setup

In [ ]:
!nvidia-smi

Sun Nov  8 00:02:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
# Install deps
%pip install --quiet rank-bm25==0.2.2 fastapi uvicorn requests psutil
cuda_ver = torch.version.cuda
cuda_tag = "cu" + cuda_ver.replace(".", "")
index_url = f"https://wheels.astral.sh/simple/{cuda_tag}/"
# Colab fallbacks to software implementations
%pip install causal-conv1d --quiet --no-build-isolation --extra-index-url {index_url}
%pip install flash-linear-attention --quiet --no-build-isolation --extra-index-url {index_url}

In [ ]:
%pip freeze > requirements.colab.txt

In [ ]:
import google.colab
google.colab.files.download("requirements.colab.txt")

## Load project and runtime config

In [ ]:
google.colab.drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import os
import subprocess
import time
import threading
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/mipt-2026f-llm-hw1-rag")
assert PROJECT.is_dir(), f"Project not found: {PROJECT}"

print("Project files:")
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(PROJECT))

Project files:
  build_index.py
  config.json
  data/corpus.jsonl
  data/dev_gold.jsonl
  data/dev_questions.jsonl
  data/extracted.md
  data/native_paragraphs.jsonl
  data/native_text.md
  data/one_question.jsonl
  data/pairs.jsonl
  data/paragraphs.jsonl
  data/textbook.pdf
  data/tfkp_15_qa.md
  experiments.ipynb
  index/bm25.pkl
  index/documents.jsonl
  index/manifest.json
  predict.py
  runtime.json
  scripts/__pycache__/tokenizer.cpython-312.pyc
  scripts/build_corpus.py
  scripts/extract_markdown.py
  scripts/generate_gold.py
  scripts/tokenizer.py


In [ ]:
%cd {PROJECT}

/content/drive/MyDrive/mipt-2026f-llm-hw1-rag


In [ ]:
print("Required files:")
for path in [
    "predict.py",
    "scripts/tokenizer.py",
    "config.json",
    "runtime.json",
]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print(f"  ok  {path}")

Required files:
  ok  predict.py
  ok  scripts/tokenizer.py
  ok  config.json
  ok  runtime.json
  ok  index/bm25.pkl
  ok  index/documents.jsonl
  ok  index/manifest.json


Building index

In [ ]:
# No explicit check for index, not bothering changing versions.
# Just rebuild every time
print("Required files for building index:")
for path in [
    "scripts/tokenizer.py",
    "config.json",
    "scripts/extract_markdown.py",
    "scripts/build_corpus.py",
]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print(f"  ok  {path}")

!python ./scripts/extract_markdown.py --pdf=data/textbook.pdf --paragraphs=data/paragraphs.jsonl --markdown=data/extracted.md
!python ./scripts/build_corpus.py --input=paragraphs.jsonl --output=data/corpus.jsonl
!python ./scripts/generate_gold.py --pairs=data/pairs.jsonl --corpus=data/corpus.jsonl

In [ ]:
print("Required index files:")
for path in [
    "config.json",
    "index/bm25.pkl",
    "index/documents.jsonl",
    "index/manifest.json",
 ]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    print(f"  ok  {path}")


### Verify Index

TODO: Actually check for manifest matches


In [ ]:
manifest = json.loads(Path("index/manifest.json").read_text(encoding="utf-8"))
tok = manifest.get("tokenization", {})
ret = manifest.get("retrieval", {})

{
    "tokenizer_version": tok.get("tokenizer_version"),
    "tokenization     ": {k: tok.get(k) for k in
      ("lowercase", "include_math_symbols", "min_token_length")},
    "document_count   ": manifest.get("document_count"),
    "avgdl            ": ret.get("avgdl"),
    "k1 / b           ": f"{ret.get("k1")}/{ret.get("b")}",
    "idf_terms        ": ret.get("idf_terms"),
}

{'tokenizer_version': None,
 'tokenization     ': {'lowercase': True,
  'include_math_symbols': True,
  'min_token_length': 1},
 'document_count   ': 937,
 'avgdl            ': 79.9787,
 'k1 / b           ': '1.5/0.75',
 'idf_terms        ': 4426}

{'tokenizer_version': None,
 'tokenization     ': {'lowercase': True,
  'include_math_symbols': True,
  'min_token_length': 1},
 'document_count   ': 937,
 'avgdl            ': 79.9787,
 'k1 / b           ': '1.5/0.75',
 'idf_terms        ': 4426}

### Load runtime config

In [ ]:
RUNTIME_PATH = PROJECT / "runtime.json"
runtime = json.loads(RUNTIME_PATH.read_text(encoding="utf-8"))

{
    "retrieval        ": runtime.get("retrieval"),
    "tokenizer_version": tok.get("tokenizer_version"),
    "tokenization     ": {k: tok.get(k) for k in
        ("lowercase", "include_math_symbols", "min_token_length")},
    "document_count   ": manifest.get("document_count"),
    "avgdl            ": ret.get("avgdl"),
    "k1 / b           ": f"{ret.get("k1")}/{ret.get("b")}",
    "idf_terms        ": ret.get("idf_terms"),
}

{'retrieval        ': {'top_k': 5},
 'tokenizer_version': None,
 'tokenization     ': {'lowercase': True,
  'include_math_symbols': True,
  'min_token_length': 1},
 'document_count   ': 937,
 'avgdl            ': 79.9787,
 'k1 / b           ': '1.5/0.75',
 'idf_terms        ': 4426}

In [ ]:
{
    "generation": runtime.get("generation"),
    "budget    ": runtime.get("budget"),
    "model     ": runtime.get("model"),
}

{'generation': {'temperature': 0.0, 'max_tokens': 1024},
 'budget    ': {'max_llm_calls_per_question': 4,
  'max_input_tokens_total': 16384,
  'max_output_tokens_total': 4096,
  'max_input_plus_output_per_call': 8192},
 'model     ': {'name': 'Qwen/Qwen3.5-4B', 'revision': '851bf6e'}}

## Load model

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_cfg = runtime.get("model", {})
MODEL_ID = model_cfg.get("name", "Qwen/Qwen3.5-4B")
REVISION = model_cfg.get("revision")

assert MODEL_ID, "runtime.model.name is required"
assert REVISION, "runtime.model.revision is required"

hf_token = os.environ.get("HF_TOKEN") \
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")

processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=REVISION, token=hf_token,
)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, revision=REVISION, device_map="auto", token=hf_token,
)

# model.config._commit_hash is missing; so cannot verify revision

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

### Model sanity check

In [ ]:
messages = [
    {
        "role": "user",
        "content": [{
            "type": "text",
            "text": "Дай определение голоморфной функции. Ответь кратко."
        }],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=750)

answer = outputs[0][inputs["input_ids"].shape[-1]:]
print(processor.decode(answer, skip_special_tokens=True))

Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Holomorphic function (голоморфная функция).
    *   Task: Define it.
    *   Constraint: Answer briefly (кратко).
    *   Language: Russian.

2.  **Identify Key Concepts:**
    *   Complex variable ($z$).
    *   Open set in the complex plane ($D$).
    *   Differentiable (complex differentiable) at every point in the set.
    *   Equivalent to being analytic (represented by a power series).

3.  **Drafting the Definition (Internal Monologue/Trial):**
    *   *Attempt 1:* Голоморфная функция — это функция комплексной переменной, которая дифференцируема в каждой точке области. (A bit too simple, but accurate).
    *   *Attempt 2:* Функция $f(z)$ называется голоморфной в области $D$, если она комплексно дифференцируема в каждой точке этой области. (Better).
    *   *Attempt 3:* Можно добавить про аналитичность, но для краткости лучше остановиться на дифференцируемости.

4.  **Refining for Brevity and Accuracy:**
    *   "Функ

## Load index

In [ ]:
import pickle

with open(PROJECT / "index/documents.jsonl", encoding="utf-8") as file:
    documents = [json.loads(line) for line in file if line.strip()]

with open(PROJECT / "index/bm25.pkl", "rb") as index_pickled:
    bm25 = pickle.load(index_pickled)

print("Total cocuments loaded:", len(documents))
print("Index type:", type(bm25).__name__)
print("Example block:")
documents[1]

Total cocuments loaded: 937
Index type: BM25Okapi
Example block:


{'block_id': 'b0000002',
 'page': 2,
 'section': '1.1. Алгебра комплексных чисел. С точки зрения разрешимости алгеб-',
 'text': '1.1. Алгебра комплексных чисел. С точки зрения разрешимости алгеб-\n\nраического уравнения x2 + 1 = 0 появляется необходимость расширения поля вещественных чисел. Пусть i новое число (мнимая единица), которое удовлетворяет условию i2 = i · i = −1. Если производить операции сложения и умножения над вещественными числами и мнимой единицей, то мы естественным образом приходим к записи комплексных чисел z = x + iy, где x, y ∈R. В этой записи x называют вещественной частью комплексного числа z и обозначают x = Re z, а y мнимой частью с обозначением y = Im z. Если x = 0, то комплексное число z называют чисто мнимым, а в случае y = 0 говорят, что z вещественно. Нуль является единственным комплексным числом, которое одновременно и вещественное и чисто мнимое. Под равенством двух комплексных чисел понимается одновременное равенство вещественных и мнимых частей. Арифме

## Local OpenAI-compatible API

Serves `processor` + `model` on localhost:8000. This adapter must always
populate `prompt_tokens` (required for the trace).

In [ ]:
import torch
import uvicorn
from fastapi import FastAPI, Request, HTTPException

app = FastAPI()

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "revision": REVISION,
        "commit_hash": getattr(model.config, "_commit_hash", None),
    }

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()

    messages = body.get("messages")
    if not isinstance(messages, list) or not messages:
        raise HTTPException(400, "messages must be a non-empty list")

    max_tokens = int(body.get("max_tokens", 700))
    if not 1 <= max_tokens <= 4096:
        raise HTTPException(400, "max_tokens must be between 1 and 4096")

    temperature = float(body.get("temperature", 0.0))
    if temperature < 0:
        raise HTTPException(400, "temperature must be >= 0")

    try:
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(model.device)

        prompt_tokens = int(inputs["input_ids"].shape[-1])

        generation_args = {
            "max_new_tokens": max_tokens,
            "do_sample": temperature > 0,
        }
        if generation_args["do_sample"]:
            generation_args["temperature"] = temperature
        if "top_p" in body:
            generation_args["top_p"] = float(body["top_p"])

        start = time.perf_counter()
        with torch.inference_mode():
            output_ids = model.generate(**inputs, **generation_args)
        elapsed = time.perf_counter() - start

        new_ids = output_ids[0, prompt_tokens:]
        completion_tokens = int(new_ids.shape[-1])
        text = processor.decode(new_ids, skip_special_tokens=True)

        return {
            "id": "qwen-completion-local",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": body.get("model", MODEL_ID),
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "length" if completion_tokens >= max_tokens else "stop",
            }],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
            "local_generation_seconds": round(elapsed, 3),
        }

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(500, f"Local model generation failed: {exc}") from exc

In [ ]:
# Guard for an idempotent start. A second run of this cell will
# not spawn a second server.
if "api_thread" not in globals() or not api_thread.is_alive(): # pyright: ignore[reportUndefinedVariable]
    server = uvicorn.Server(
        uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    )
    api_thread = threading.Thread(target=server.run, daemon=True)
    api_thread.start()

import requests
for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print(r.json())
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Local Qwen API did not start")

{'status': 'ok', 'model': 'Qwen/Qwen3.5-4B', 'revision': '851bf6e', 'commit_hash': None}


### API sanity check

In [ ]:
import requests

r = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{
            "role": "user",
            "content": "Return exactly the following JSON: {\"ok\": true}"
        }],
        "temperature": 0.0,
        "max_tokens": 80,
    },
    timeout=180,
)

print("HTTP:", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))
# TODO: add assert
r.raise_for_status()

HTTP: 200
{
  "id": "qwen-completion-local",
  "object": "chat.completion",
  "created": 1794097658,
  "model": "Qwen/Qwen3.5-4B",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "{\"ok\": true}\n"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 22,
    "completion_tokens": 8,
    "total_tokens": 30
  },
  "local_generation_seconds": 0.761
}


Set environment for the `predict.py` script

In [ ]:
os.environ["LLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
os.environ["LLM_MODEL"] = MODEL_ID
os.environ["SUBMISSION_ID"] = "local-dev"
os.environ.pop("LLM_API_KEY", None)

## Experiment runner

In [ ]:
from google.colab import files


def run_predict(experiment: str, questions: str,
                answers: str, trace: str) -> None:
    cmd = [
        "python", "predict.py",
        "--experiment-name", experiment,
        "--index", "./index",
        "--questions", questions,
        "--output", answers,
        "--trace", trace,
        "--config", "./config.json",
        "--runtime", "./runtime.json",
    ]
    print("$", " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout:
        for line in process.stdout:
            print(line, end="")

    process.wait()

    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)


def download_all(*prefixes: str) -> None:
    for p in prefixes:
        for suffix in ("jsonl",):
            path = f"{p}.{suffix}"
            if Path(path).exists():
                files.download(path)

### Reference questions - `full`

In [ ]:
run_predict(
    "full",
    "./data/dev_questions.jsonl",
    "./dev_answers.jsonl",
    "./dev_trace.jsonl",
)
download_all("dev_answers_full", "dev_trace_full")

$ python predict.py --experiment-name full --index ./index --questions ./data/dev_questions.jsonl --output ./dev_answers.jsonl --trace ./dev_trace.jsonl --config ./config.json --runtime ./runtime.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Reference questions - `no_context`

Note: output of deliberately violates the spec schema.
(`abstain=false` with empty `citations`). I guess citation F1 is ~0 by
construction and score dip is expected.

In [ ]:
run_predict(
    "no_context",
    "./data/dev_questions.jsonl",
    "./dev_answers.jsonl",
    "./dev_trace.jsonl",
)
download_all("dev_answers_no_context", "dev_trace_no_context")

### Reference questions - `rag_single`

In [ ]:
run_predict(
    "rag_single",
    "./data/dev_questions.jsonl",
    "./dev_answers.jsonl",
    "./dev_trace.jsonl",
)
download_all("dev_answers_rag_single", "dev_trace_rag_single")

### Reference questions - `ablation_no_rerank`

BM25 selects top_k, blocks reordered by corpus position.

In [ ]:
run_predict(
    "ablation_no_rerank",
    "./data/dev_questions.jsonl",
    "./dev_answers.jsonl",
    "./dev_trace.jsonl",
)
download_all("dev_answers_ablation_no_rerank", "dev_trace_ablation_no_rerank")

### Reference questions - `ablation_top1`

In [ ]:
run_predict(
    "ablation_top1",
    "./data/dev_questions.jsonl",
    "./dev_answers.jsonl",
    "./dev_trace.jsonl",
)
download_all("dev_answers_ablation_top1", "dev_trace_ablation_top1")

### Own questions - `full`

In [ ]:
if Path("./data/own_questions.jsonl").exists():
    run_predict(
        "full",
        "./data/own_questions.jsonl",
        "./own_answers.jsonl",
        "./own_trace.jsonl",
    )
    download_all("own_answers_full", "own_trace_full")
else:
    print("Skipping: ./data/own_questions.jsonl not present")

$ python predict.py --experiment-name full --index ./index --questions ./data/own_questions.jsonl --output ./own_answers.jsonl --trace ./own_trace.jsonl --config ./config.json --runtime ./runtime.json
NOTE: runtime.model.revision='851bf6e' is not verifiable client-side. The inference server must load the model with revision pinned and log model.config._commit_hash.
  retrieval: policy=bm25_ranked top_k=5 ids=['b0000459', 'b0000178', 'b0000260', 'b0000161', 'b0000810']
  retrieval: policy=bm25_ranked top_k=5 ids=['b0000329', 'b0000321', 'b0000330', 'b0000334', 'b0000332']
  retrieval: policy=bm25_ranked top_k=5 ids=['b0000260', 'b0000293', 'b0000198', 'b0000161', 'b0000263']
  retrieval: policy=bm25_ranked top_k=5 ids=['b0000341', 'b0000339', 'b0000700', 'b0000681', 'b0000124']
  retrieval: policy=bm25_ranked top_k=5 ids=['b0000350', 'b0000345', 'b0000356', 'b0000346', 'b0000495']
  retrieval: policy=bm25_ranked top_k=5 ids=['b0000601', 'b0000356', 'b0000198', 'b0000397', 'b0000777']
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Usage statistics

Quick glance before an evaluation

In [ ]:
import statistics

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

trace_path = Path.cwd() / "own_trace_full.jsonl"
if not trace_path.exists():
    print(f"Skipping: {trace_path.name} not found")
else:
    answers = read_jsonl(Path.cwd() / "own_answers_full.jsonl")
    traces = read_jsonl(trace_path)
    if not traces:
        raise ValueError(f"No trace records in {trace_path}")

    input_tokens  = [trace.get("input_tokens", 0) for trace in traces]
    output_tokens = [trace.get("output_tokens", 0) for trace in traces]
    llm_calls     = [trace.get("llm_calls", 0) for trace in traces]
    latencies     = [trace.get("elapsed_ms", 0) for trace in traces]

    # cost score model from the spec
    cost_score = [
        t.get("input_tokens", 0)
        + 4 * t.get("output_tokens", 0)
        + 256 * t.get("llm_calls", 0)
        for t in traces
    ]

    answer_ids = {str(a["id"]) for a in answers}
    trace_ids  = {str(t["id"]) for t in traces}

    print("Overall:")
    print(f"{'answers':<32}: {len(answers)}")
    print(f"{'trace records':<32}: {len(traces)}")
    print(f"{'trace missing':<32}: {len(answer_ids - trace_ids)}")
    print(f"{'answer missing':<32}: {len(trace_ids - answer_ids)}\n")

    print("Tokens and calls:")
    print(f"{'input tokens, total':<32}: {sum(input_tokens):,}")
    print(f"{'input tokens, mean':<32}: {statistics.mean(input_tokens):.1f}")
    print(f"{'output tokens, total':<32}: {sum(output_tokens):,}")
    print(f"{'output tokens, mean':<32}: {statistics.mean(output_tokens):.1f}")
    print(f"{'llm calls, total':<32}: {sum(llm_calls)}")
    print(f"{'llm calls, mean':<32}: {statistics.mean(llm_calls):.2f}\n")

    print("Cost score:")
    print(f"{'total'}: {sum(cost_score):,}")
    print(f"{'mean'}: {statistics.mean(cost_score):.1f}\n")

    print("Latency (ms):")
    print(f"{'median':<32}: {statistics.median(latencies):.0f}")
    print(f"{'mean':<32}: {statistics.mean(latencies):.0f}\n")


Overall:
answers                         : 8
trace records                   : 8
trace missing                   : 0
answer missing                  : 0

Tokens and calls:
input tokens, total             : 10,355
input tokens, mean              : 1294.4
output tokens, total            : 868
output tokens, mean             : 108.5
llm calls, total                : 8
llm calls, mean                 : 1.00

Cost score:
total: 15,875
mean: 1984.4

Latency (ms):
median                          : 10669
mean                            : 11813



## Evaluation

In [ ]:
from evaluate import evaluate

results = evaluate(
    gold_path="./data/dev_gold.jsonl",
    answers_dir=".",
    traces_dir=".",
    experiments=["full", "no_context", "rag_single",
                 "ablation_no_rerank", "ablation_top1"],
    baseline="rag_single",
    csv_out="./experiments.csv",
    details_out="./details.jsonl",
)

from google.colab import files
files.download("experiments.csv")
files.download("details.jsonl")